In [2]:
import pandas as pd
import json
import os
import logging
import sys

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', stream=sys.stdout)

# --- CONFIGURATION ---
# The master file containing all 22 data points per function (Rounds 0-11)
INPUT_MASTER_FILE = 'bbo_master_w13.csv'
OUTPUT_DIR = 'add_data'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'week13_clean_inputs.json')

# Dimensionality Map
FUNCTION_DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}

def generate_final_submission():
    if not os.path.exists(INPUT_MASTER_FILE):
        logging.error(f"Input file {INPUT_MASTER_FILE} not found. Please run create_bbo_master_w13.ipynb first.")
        return

    df_master = pd.read_csv(INPUT_MASTER_FILE)
    logging.info(f"Loaded {len(df_master)} historical data points.")
    
    final_queries = []

    logging.info("\n--- FINAL SUBMISSION STRATEGY: EXPERIENCE REPLAY (ARGMAX Y) ---")

    for f_id in range(1, 9):
        dim = FUNCTION_DIMS[f_id]
        # Filter for the specific function
        df_f = df_master[df_master['Function ID'] == f_id].copy()
        
        # Find the row with the absolute maximum Y value seen so far
        best_row_idx = df_f['Y'].idxmax()
        best_row = df_f.loc[best_row_idx]
        
        best_y = best_row['Y']
        
        # Extract the X coordinates corresponding to that best Y
        x_cols = [f'X{i+1}' for i in range(dim)]
        best_x = best_row[x_cols].tolist()
        
        # --- CORRECTION APPLIED HERE ---
        # Convert to float and explicitly round to 6 decimal places to fix floating point artifacts
        best_x_clean = [float(round(x, 6)) for x in best_x]
        
        final_queries.append(best_x_clean)
        
        logging.info(f"F{f_id}: Best Historical Y = {best_y:.6f} found in Round {int(best_row.get('Round', -1))}")
        logging.info(f"      Re-submitting Coordinates: {best_x_clean}")

    # Save outputs to JSON
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(final_queries, f, indent=4)
    
    logging.info(f"\nSUCCESS: Generated final queries based on historical bests.")
    logging.info(f"Saved to {OUTPUT_FILE}")

if __name__ == '__main__':
    generate_final_submission()


INFO: Loaded 176 historical data points.
INFO: 
--- FINAL SUBMISSION STRATEGY: EXPERIENCE REPLAY (ARGMAX Y) ---
INFO: F1: Best Historical Y = 0.174350 found in Round 4
INFO:       Re-submitting Coordinates: [0.598715, 0.631376]
INFO: F2: Best Historical Y = 0.728198 found in Round 11
INFO:       Re-submitting Coordinates: [0.684632, 0.968359]
INFO: F3: Best Historical Y = -0.010130 found in Round 11
INFO:       Re-submitting Coordinates: [0.616525, 0.739696, 0.508262]
INFO: F4: Best Historical Y = 0.494016 found in Round 8
INFO:       Re-submitting Coordinates: [0.428378, 0.369407, 0.345557, 0.399058]
INFO: F5: Best Historical Y = 8662.405001 found in Round 8
INFO:       Re-submitting Coordinates: [1.0, 1.0, 1.0, 1.0]
INFO: F6: Best Historical Y = -0.255485 found in Round 8
INFO:       Re-submitting Coordinates: [0.510625, 0.33055, 0.518944, 0.77119, 0.154218]
INFO: F7: Best Historical Y = 1.648730 found in Round 2
INFO:       Re-submitting Coordinates: [0.55198, 0.11195, 0.70617, 0.45